# Reconnaissance d'expressions faciales — FER2013

**Projet Deep Learning** — Nikko et Simon

**Objectif :** entraîner et comprendre un modèle qui reconnaît 7 expressions faciales
(angry, disgust, fear, happy, neutral, sad, surprise) à partir d'images de visages 48×48 en niveaux de gris.

**Dataset :** FER2013 — Kaggle `msambare/fer2013`

**Plan du notebook** (construit progressivement, une cellule par micro-étape) :
1. Exploration et préparation du dataset
2. Baseline dense
3. CNN
4. Entraînement
5. Évaluation et analyse des erreurs
6. Expériences (≥ 3)


### Étape 0 — Configuration commune et téléchargement des données

Cette cellule est lancée **en premier** ; toutes les suivantes en dépendent.

1. **Racine du projet.** En local, c'est le dossier qui contient `.gitignore` : le notebook marche qu'on le lance depuis la racine ou depuis `notebooks/`. Sur Colab, si le dépôt n'a pas été cloné, il n'y a pas de `.gitignore` : on reste alors dans le dossier courant (`/content`).
2. **Constantes partagées.** `DATA_DIR`, `CLASS_NAMES`, `IMAGE_SIZE` et `SEED` sont définies **une seule fois ici** et réutilisées partout : changer une valeur à un endroit la change pour tout le notebook.
3. **Graine aléatoire (*seed*).** `keras.utils.set_random_seed(42)` fixe le hasard de Python, NumPy et TensorFlow (découpe train/validation, poids initiaux, mélange des lots). Deux expériences ne diffèrent alors que par ce qu'on a volontairement changé — indispensable pour comparer nos ≥ 3 expériences. (Sur GPU, quelques opérations restent légèrement non déterministes : de petites variations restent possibles.)
4. **Téléchargement idempotent.** `data/` n'est pas dans Git (≈ 200 Mo). Si `data/fer2013/train` et `data/fer2013/test` existent déjà, la cellule ne télécharge rien ; sinon elle récupère FER2013 avec `kagglehub` (bibliothèque officielle de Kaggle). La relancer 10 fois donne le même résultat qu'une seule fois.

In [ ]:
# Étape 0 : configuration commune + téléchargement de FER2013 s'il manque
import os
import shutil
from pathlib import Path

from tensorflow import keras

# --- 1) Racine du projet ---
# On remonte les dossiers jusqu'à trouver .gitignore (racine du dépôt).
racine = Path.cwd()
while racine != racine.parent and not (racine / ".gitignore").exists():
    racine = racine.parent
if not (racine / ".gitignore").exists():
    # Pas de dépôt (ex. notebook ouvert seul sur Colab) : on reste dans le dossier courant.
    racine = Path.cwd()
os.chdir(racine)  # les cellules suivantes utilisent des chemins relatifs à cette racine
print("Racine du projet :", racine)

# --- 2) Constantes partagées par tout le notebook ---
DATA_DIR = Path("data/fer2013")
# Cet ordre fixe la position du 1 dans le vecteur one-hot :
# 0 = angry, 1 = disgust, 2 = fear, 3 = happy, 4 = neutral, 5 = sad, 6 = surprise.
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
IMAGE_SIZE = (48, 48)  # taille des images FER2013 (vérifiée à l'étape 1b)
SEED = 42

# --- 3) Graine aléatoire : rend les résultats reproductibles ---
keras.utils.set_random_seed(SEED)

# --- 4) Téléchargement, seulement si les données manquent ---
if (DATA_DIR / "train").is_dir() and (DATA_DIR / "test").is_dir():
    print(f"Données déjà présentes dans {DATA_DIR} — aucun téléchargement.")
else:
    import kagglehub  # déjà installé sur Colab ; en local : pip install kagglehub

    print("Téléchargement du dataset msambare/fer2013 via kagglehub…")
    source = Path(kagglehub.dataset_download("msambare/fer2013"))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "test"]:
        if not (DATA_DIR / split).exists():  # jamais réécrit si déjà copié
            shutil.copytree(source / split, DATA_DIR / split)
    print("Dataset copié dans", DATA_DIR.resolve())

## Partie 1 — Recherche, compréhension et préparation des données

### Description du dataset : FER2013

| Élément | Valeur |
|---|---|
| **Source** | Kaggle, [`msambare/fer2013`](https://www.kaggle.com/datasets/msambare/fer2013) (version au format images, un dossier par classe) |
| **Auteurs** | Pierre-Luc Carrier et Aaron Courville. Dataset publié pour le concours *Challenges in Representation Learning* (ICML 2013) : Goodfellow et al., 2013, [arXiv:1307.0414](https://arxiv.org/abs/1307.0414) |
| **Licence** | Database Contents License (DbCL) v1.0, indiquée sur la page Kaggle. Les photos d'origine viennent du web : on utilise le dataset dans un cadre académique |
| **Collecte** | Images trouvées via Google Images avec des mots-clés d'émotions ; visages détectés et recadrés automatiquement, vérifiés par des annotateurs, puis réduits en 48 × 48 niveaux de gris |
| **Nombre d'images** | 35 887 : 28 709 en `train`, 7 178 en `test` |
| **Classes** | 7 : `angry` (colère), `disgust` (dégoût), `fear` (peur), `happy` (joie), `neutral` (neutre), `sad` (tristesse), `surprise` |
| **Dimensions** | 48 × 48 pixels |
| **Format** | Niveaux de gris (1 canal), fichiers `.jpg` |
| **Déséquilibre** | Oui : 436 images `disgust` contre 7 215 `happy` en `train` (≈ 16,5 fois moins) |

**Pourquoi ce dataset ?** Il est public, très utilisé comme référence, et ses visages sont **déjà extraits et recadrés** : c'est exactement l'entrée « visage déjà extrait → CNN → expression » des Parties 1 à 6.

Les chiffres du tableau sont mesurés par les cellules 1a et 1b ci-dessous.

### Étape 1a — Compter les images et visualiser le déséquilibre

Avant de charger les données, on compte les images de chaque dossier : chaque dossier de classe sert d'étiquette (label), c'est cette structure que Keras lira. Le graphique montre le **déséquilibre entre les classes** : un modèle aura tendance à mieux reconnaître les classes fréquentes, c'est pourquoi on regardera le rappel par classe en Partie 5, pas seulement l'accuracy.

In [ ]:
# Micro-étape 1a : compter les images de chaque classe, puis visualiser le déséquilibre
import matplotlib.pyplot as plt

nombres_train = []  # nombre d'images d'entraînement par classe, dans l'ordre de CLASS_NAMES

# Chaque dossier de classe (data/fer2013/train/happy, ...) contient les images de cette classe :
# le nom du dossier EST l'étiquette.
for split in ["train", "test"]:
    total = 0
    for classe in CLASS_NAMES:
        dossier = DATA_DIR / split / classe
        n = len(list(dossier.glob("*.jpg")))  # nombre de fichiers .jpg dans le dossier
        total = total + n
        print(f"{split}/{classe} : {n}")
        if split == "train":
            nombres_train.append(n)
    print(f"{split} total : {total}")
    print()

# --- graphique : une barre par classe (même ordre que CLASS_NAMES) ---
ratio = max(nombres_train) / min(nombres_train)

fig, ax = plt.subplots(figsize=(8, 4))
barres = ax.barh(CLASS_NAMES, nombres_train, height=0.6, color="#2a78d6")
ax.bar_label(barres, padding=4)  # le nombre d'images écrit au bout de chaque barre
ax.invert_yaxis()                # angry en haut, comme dans CLASS_NAMES
ax.set_xlabel("Nombre d'images (train)")
ax.spines[["top", "right"]].set_visible(False)  # axes plus discrets
ax.set_title(f"Images d'entraînement par classe : la plus fréquente en a {ratio:.1f}× plus que la plus rare")
plt.tight_layout()
plt.show()

### Étape 1b — Regarder les images et vérifier le format

**Pourquoi ?** Un modèle apprend à partir de pixels : s'il reçoit des images de tailles ou de modes différents, la couche `Flatten` produira des vecteurs de longueurs différentes et l'entraînement échouera (ou buggera silencieusement).

On fait donc deux choses :
1. **On affiche une grille de 7 classes × 5 images** tirées de `data/fer2013/train/<classe>/` : on vérifie *à l'œil* que les dossiers contiennent bien des visages et que les classes sont cohérentes (utile aussi pour l'oral : montrer des exemples réels).
2. **On vérifie le format sur *toutes* les images** (train + test) avec PIL : taille `(48, 48)`, mode `L` (niveaux de gris) et extension `.jpg`. Un `Counter` (bibliothèque standard Python) compte combien de fois chaque valeur apparaît ; on liste les exceptions plutôt que de les cacher.

In [ ]:
# Micro-étape 1b : exemples d'images + vérification du format (taille, mode, extension)
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

N_EXEMPLES = 5  # nombre d'images affichées par classe

# --- 1) grille 7 lignes (classes) x 5 colonnes (exemples) -------------------
fig, axes = plt.subplots(len(CLASS_NAMES), N_EXEMPLES, figsize=(10, 14))
for ligne, classe in enumerate(CLASS_NAMES):
    # on trie les fichiers pour que la grille soit la même à chaque exécution
    fichiers = sorted((DATA_DIR / "train" / classe).glob("*.jpg"))[:N_EXEMPLES]
    for col, chemin in enumerate(fichiers):
        with Image.open(chemin) as img:  # le `with` ferme l'image automatiquement
            axes[ligne, col].imshow(img, cmap="gray")  # images en niveaux de gris
        axes[ligne, col].set_xticks([])  # on masque les graduations : on veut voir l'image
        axes[ligne, col].set_yticks([])
    # nom de la classe écrit à gauche de la ligne
    axes[ligne, 0].set_ylabel(classe, rotation=0, ha="right", va="center")

fig.suptitle("FER2013 — 5 exemples par classe (split train)", fontsize=14)
plt.tight_layout()
plt.show()

# --- 2) vérification du format sur TOUTES les images (train + test) --------
# Counter compte les valeurs rencontrées, ex. {(48, 48): 35887}
tailles, modes, extensions = Counter(), Counter(), Counter()
anomalies = []  # (chemin, description du problème)
n_images = 0

for split in ["train", "test"]:
    for classe in CLASS_NAMES:
        dossier = DATA_DIR / split / classe
        for chemin in sorted(dossier.iterdir()):  # tous les fichiers du dossier de classe
            n_images = n_images + 1
            extension = chemin.suffix.lower()  # ".jpg", ".png", ...
            try:
                with Image.open(chemin) as img:
                    taille = img.size
                    mode = img.mode
            except OSError:  # fichier corrompu ou illisible : on le note et on continue
                anomalies.append((chemin, "illisible"))
                continue
            tailles[taille] += 1
            modes[mode] += 1
            extensions[extension] += 1
            if taille != IMAGE_SIZE or mode != "L" or extension != ".jpg":
                anomalies.append((chemin, f"taille {taille}, mode {mode}, extension {extension}"))

print(f"Images vérifiées : {n_images} (train + test)")
print(f"Tailles observées    : {dict(tailles)}")
print(f"Modes PIL observés   : {dict(modes)}")
print(f"Extensions observées : {dict(extensions)}")
if anomalies:
    print(f"\n{len(anomalies)} exception(s) trouvée(s) sur {n_images} images :")
    for chemin, probleme in anomalies[:20]:  # on n'affiche que les 20 premières
        print(f"  - {chemin} -> {probleme}")
else:
    print(f"\nFormat : 48×48, mode L, .jpg — vérifié sur {n_images} images, aucune exception.")

### Étape 1c — Construire les datasets d'entraînement, de validation et de test

**Pourquoi ?** Jusqu'ici on a seulement *regardé* les fichiers. Le modèle a besoin d'un `tf.data.Dataset` : un flux de lots d'images + d'étiquettes, prêt pour `model.fit()`.

Cinq points à comprendre :

1. **Normalisation** — les pixels sont des entiers entre 0 et 255. On les divise par 255 pour les ramener dans `[0, 1]` : des entrées petites et à la même échelle rendent l'entraînement plus stable et, en général, plus rapide.
2. **Encodage one-hot** — `label_mode="categorical"` transforme l'étiquette `"happy"` en vecteur `[0, 0, 0, 1, 0, 0, 0]` de 7 valeurs (la position du 1 suit l'ordre de `CLASS_NAMES`). C'est la forme attendue par une sortie `softmax` avec la perte `categorical_crossentropy`.
3. **Validation** — 20 % de `data/fer2013/train` est mis de côté (5 741 images) pour mesurer la généralisation pendant l'entraînement et repérer le surapprentissage. Ces images n'entraînent jamais les poids : elles servent seulement à décider quand s'arrêter (`early stopping`, choix du meilleur modèle).
4. **Test** — `data/fer2013/test` (7 178 images) reste intact, sans mélange ni découpe : il est réservé à l'évaluation finale, une seule fois, à la toute fin du projet. Si on l'utilisait pendant le développement, on « tricherait » sans le savoir.

Rappel de l'étape 1b : toutes les images font déjà 48×48 en niveaux de gris, donc la forme d'une image est `(48, 48, 1)` et aucune redimensionnement n'est nécessaire.

5. **Performance** — `.prefetch()` prépare le lot suivant pendant que le modèle calcule sur le lot courant. `.cache()` garde les lots en mémoire après la première lecture : on l'utilise **seulement pour la validation et le test**. Sur l'entraînement, il figerait l'ordre des images de la première époque ; or on veut que les images soient **mélangées différemment à chaque époque**, pour que le modèle ne s'habitue pas à un ordre fixe.

In [ ]:
# Micro-étape 1c : construire les datasets TensorFlow (train / validation / test)
import tensorflow as tf

BATCH_SIZE = 64         # nombre d'images par lot
VALIDATION_SPLIT = 0.2  # 20 % du dossier train est réservé à la validation

# --- 1) train + validation : une seule passe sur data/fer2013/train ---
# subset="both" renvoie directement un tuple (train_ds, val_ds).
# seed=SEED rend la découpe reproductible : la même image tombe toujours du même côté.
# color_mode="grayscale"  -> 1 seul canal, donc une forme (48, 48, 1).
# label_mode="categorical" -> étiquette encodée en one-hot (vecteur de 7 valeurs).
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    subset="both",
    seed=SEED,
)

# --- 2) test : data/fer2013/test, utilisé tel quel, sans découpe ---
# shuffle=False garde l'ordre des fichiers, pour une évaluation stable et reproductible.
test_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# --- 3) nombre d'images réellement présent dans chaque ensemble ---
# .file_paths n'existe que sur le dataset d'origine : on le lit avant le .map ci-dessous.
print("Images en train      :", len(train_ds.file_paths))  # attendu : 22968
print("Images en validation :", len(val_ds.file_paths))    # attendu : 5741
print("Images en test       :", len(test_ds.file_paths))   # attendu : 7178

# --- 4) normalisation : les pixels passent de l'échelle [0, 255] à [0, 1] ---
def normaliser(images, labels):
    return images / 255.0, labels

# .map applique la fonction à chaque lot, au fur et à mesure de la lecture.
# .prefetch prépare le lot suivant pendant que le modèle calcule sur le lot courant.
# .cache garde les lots en mémoire : seulement pour validation et test, car sur train
# il figerait l'ordre des images de la 1re époque (plus de mélange entre les époques).
train_ds = train_ds.map(normaliser).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(normaliser).cache().prefetch(tf.data.AUTOTUNE)

# --- 5) contrôle : forme d'un lot et valeurs des pixels ---
images_batch, labels_batch = next(iter(train_ds))  # premier lot : 64 images + étiquettes
print("Forme du lot d'images  :", images_batch.shape)  # attendu : (64, 48, 48, 1)
print("Forme du lot de labels :", labels_batch.shape)  # attendu : (64, 7)
print("Valeur min des pixels  :", float(images_batch.numpy().min()))  # entre 0 et 1
print("Valeur max des pixels  :", float(images_batch.numpy().max()))  # entre 0 et 1

## Partie 2 — Baseline dense

**Pourquoi une baseline ?** Avant de construire un CNN, on entraîne le modèle le plus simple possible. Son score sert de **point de comparaison** : si le CNN ne fait pas nettement mieux, c'est qu'il y a un problème. Sans baseline, impossible de dire si 55 % est un bon ou un mauvais résultat.

**Le modèle : un réseau entièrement connecté (*dense*)**, selon le schéma de l'énoncé `Image → Flatten → Dense + activation → Dense → Softmax`.
1. `Flatten` met les 48 × 48 = **2 304 pixels** de l'image bout à bout, dans un seul vecteur.
2. Une couche cachée `Dense(128, activation="relu")` : chacun des 128 neurones est relié à **toutes** les 2 304 valeurs d'entrée. L'activation `relu` (`max(0, x)`) permet au réseau d'apprendre des relations non linéaires.
3. Une couche de sortie `Dense(7, activation="softmax")` donne **7 probabilités qui somment à 1**, une par expression. La classe prédite est celle qui a la plus grande probabilité.

**Sa limite (attendue) :** après `Flatten`, le réseau ne sait plus quels pixels sont voisins. Un sourire décalé de quelques pixels devient un vecteur complètement différent. C'est exactement ce que le CNN de la Partie 3 corrigera.

**Compilation :**
- `loss="categorical_crossentropy"` : la perte adaptée à des étiquettes one-hot et une sortie softmax (étape 1c). Elle est faible quand le modèle donne une forte probabilité à la bonne classe.
- `optimizer="adam"` : l'algorithme qui ajuste les poids pour faire baisser la perte (une version améliorée de la descente de gradient).
- `metrics=["accuracy"]` : le **taux de bonnes réponses**, plus facile à lire que la perte.

In [ ]:
# Partie 2a : définir et compiler la baseline dense
from tensorflow.keras import layers

NUM_CLASSES = len(CLASS_NAMES)  # 7 expressions

baseline = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),  # (48, 48) + (1,) = (48, 48, 1) : une image, 1 canal
        layers.Flatten(),                       # 48 x 48 x 1 = 2304 valeurs à la suite
        layers.Dense(128, activation="relu"),   # couche cachée : 128 neurones
        layers.Dense(NUM_CLASSES, activation="softmax"),  # 7 probabilités qui somment à 1
    ],
    name="baseline_dense",
)

baseline.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# summary() affiche chaque couche et son nombre de paramètres (poids à apprendre).
# Exemple : la couche cachée a 2304 x 128 poids + 128 biais = 295 040 paramètres.
baseline.summary()

### Comment le réseau apprend : poids, propagation avant, perte, rétropropagation

**1. L'entrée.** L'image 48 × 48 devient, après `Flatten`, un vecteur $x$ de 2 304 nombres entre 0 et 1 (un par pixel).

**2. Poids et biais.** Chaque neurone calcule une somme pondérée de ses entrées :

$$z = w_1 x_1 + w_2 x_2 + \dots + w_{2304}\, x_{2304} + b$$

Les **poids** $w$ disent quelle importance donner à chaque pixel ; le **biais** $b$ décale le résultat. Ce sont les seuls nombres que le réseau apprend : ici 295 040 + 903 = **295 943 paramètres**.

**3. Propagation avant (*forward pass*).** L'image traverse le réseau couche par couche :
image → `Flatten` → 128 neurones + ReLU → 7 neurones → softmax → 7 probabilités.

**4. Fonctions d'activation.**
- **ReLU** dans la couche cachée : $\text{ReLU}(z) = \max(0, z)$. Sans fonction non linéaire, empiler des couches reviendrait à une seule couche linéaire, incapable d'apprendre des formes complexes.
- **Softmax** en sortie : $\text{softmax}(z_i) = \dfrac{e^{z_i}}{\sum_j e^{z_j}}$ transforme les 7 scores en probabilités positives qui somment à 1. La classe prédite est celle de plus grande probabilité.

**5. La fonction de perte.** La *categorical cross-entropy* vaut $L = -\log(p_{\text{vraie classe}})$ : si le modèle donne 0,72 à la bonne classe, $L \approx 0{,}33$ ; s'il ne lui donne que 0,05, $L \approx 3{,}0$. Plus la perte est basse, meilleur est le modèle.

**6. Rétropropagation (*backpropagation*).** Après chaque lot, on calcule pour **chaque poids** le gradient $\frac{\partial L}{\partial w}$ : de combien la perte change si on modifie un peu ce poids. Le calcul part de la sortie et remonte vers l'entrée, couche par couche, avec la règle de dérivation en chaîne.

**7. Descente de gradient.** Chaque poids est déplacé dans la direction qui fait baisser la perte :

$$w \leftarrow w - \eta \, \frac{\partial L}{\partial w}$$

$\eta$ est le **taux d'apprentissage** (*learning rate*, 0,001 par défaut avec Adam). On fait une mise à jour par lot de 64 images, soit environ 359 mises à jour par époque (22 968 / 64). **Adam** est une variante de la descente de gradient qui adapte automatiquement le pas de chaque poids.

**8. Sortie multiclasse.** 7 classes → 7 neurones de sortie + softmax (pour 2 classes, on aurait 1 neurone + sigmoïde).

### 2b — Entraîner la baseline

`model.fit()` répète l'apprentissage pendant `EPOCHS_BASELINE` **époques** (une époque = le modèle a vu une fois toutes les images d'entraînement, lot par lot).

À la fin de chaque époque, Keras affiche :
- `loss` / `accuracy` : mesurées sur les images d'**entraînement** ;
- `val_loss` / `val_accuracy` : mesurées sur la **validation**, des images que le modèle n'apprend jamais.

Ce qu'on surveille : si `accuracy` continue de monter alors que `val_accuracy` stagne ou baisse, le modèle **apprend par cœur** les images d'entraînement au lieu de généraliser : c'est le **surapprentissage** (*overfitting*). On garde ici un nombre d'époques fixe pour bien voir ce phénomène sur les courbes ; l'arrêt automatique (*early stopping*) viendra en Partie 4.

In [ ]:
# Partie 2b : entraîner la baseline (le jeu de test n'est PAS utilisé ici)
EPOCHS_BASELINE = 20

historique_baseline = baseline.fit(
    train_ds,
    validation_data=val_ds,  # mesuré à chaque époque, sans jamais modifier les poids
    epochs=EPOCHS_BASELINE,
)

### 2c — Courbes d'apprentissage et score de la baseline

On trace la perte et l'accuracy époque par époque, pour l'entraînement et la validation, puis on donne le score final sur la **validation**.

Deux repères pour juger ce score :
- **le hasard** : répondre une classe au hasard donne 1 chance sur 7, soit environ 14 % ;
- **la classe majoritaire** : répondre toujours `happy` (la classe la plus fréquente) donne environ 25 %. Un modèle qui ne dépasse pas ce chiffre n'a rien appris d'utile, même si 25 % paraît mieux que le hasard.

In [ ]:
# Partie 2c : courbes d'apprentissage + score de la baseline sur la validation
import matplotlib.pyplot as plt
import numpy as np

# historique_baseline.history est un dictionnaire : une liste de valeurs par métrique, une valeur par époque
courbes = historique_baseline.history
epoques = range(1, len(courbes["loss"]) + 1)

fig, (ax_perte, ax_accuracy) = plt.subplots(1, 2, figsize=(12, 4))

ax_perte.plot(epoques, courbes["loss"], label="entraînement")
ax_perte.plot(epoques, courbes["val_loss"], label="validation")
ax_perte.set_title("Perte (plus bas = mieux)")
ax_perte.set_xlabel("Époque")
ax_perte.legend()

ax_accuracy.plot(epoques, courbes["accuracy"], label="entraînement")
ax_accuracy.plot(epoques, courbes["val_accuracy"], label="validation")
ax_accuracy.set_title("Accuracy (plus haut = mieux)")
ax_accuracy.set_xlabel("Époque")
ax_accuracy.legend()

fig.suptitle("Baseline dense — courbes d'apprentissage")
plt.tight_layout()
plt.show()

# Score final sur la validation (les poids ne changent pas pendant evaluate)
perte_val, accuracy_val = baseline.evaluate(val_ds, verbose=0)

# Repère "classe majoritaire" : on compte les étiquettes one-hot de la validation
comptes = np.zeros(NUM_CLASSES)
for images, labels in val_ds:
    comptes = comptes + labels.numpy().sum(axis=0)  # somme des vecteurs one-hot = nombre d'images par classe
classe_majoritaire = CLASS_NAMES[int(comptes.argmax())]
accuracy_majoritaire = comptes.max() / comptes.sum()

print(f"Baseline dense — accuracy sur la validation : {accuracy_val:.1%}")
print(f"Repère hasard (1 chance sur {NUM_CLASSES}) : {1 / NUM_CLASSES:.1%}")
print(f"Repère « toujours {classe_majoritaire} » : {accuracy_majoritaire:.1%}")

### 2d — Ce que le réseau renvoie pour une image

Pour une image de validation, on affiche les **7 probabilités** produites par softmax, leur somme (= 1) et la classe prédite (la plus probable), à comparer avec la vraie classe.

In [ ]:
# Partie 2d : les 7 probabilités prédites pour UNE image de validation
images_val, labels_val = next(iter(val_ds))  # un lot de 64 images de validation

# predict attend un lot d'images : images_val[:1] est un lot qui contient seulement la 1re image
probabilites = baseline.predict(images_val[:1], verbose=0)[0]  # 7 probabilités
vraie_classe = CLASS_NAMES[int(labels_val[0].numpy().argmax())]
classe_predite = CLASS_NAMES[int(probabilites.argmax())]

plt.figure(figsize=(2.5, 2.5))
plt.imshow(images_val[0], cmap="gray", vmin=0, vmax=1)  # pixels normalisés entre 0 et 1
plt.title(f"vraie classe : {vraie_classe}")
plt.axis("off")
plt.show()

for classe, probabilite in zip(CLASS_NAMES, probabilites):
    print(f"{classe:10s} {probabilite:.2f}")
print(f"Somme des probabilités : {probabilites.sum():.2f}")
print(f"Classe prédite : {classe_predite} (vraie classe : {vraie_classe})")

## Partie 3 — CNN

**Le modèle : un petit réseau convolutif**, selon le schéma de l'énoncé `Image → Conv2D + ReLU → MaxPooling → Conv2D + ReLU → MaxPooling → Flatten → Dense → Softmax`.
1. `Conv2D(32, 3)` fait glisser 32 **filtres** de 3 × 3 pixels sur l'image. Chaque filtre apprend à détecter un petit motif local (un bord, un coin) **où qu'il soit** dans l'image : c'est ce qui manquait à la baseline dense.
2. `MaxPooling2D(2)` ne garde que la plus grande valeur de chaque carré de 2 × 2 : la carte est deux fois plus petite et la détection supporte mieux un petit décalage du visage.
3. Un second bloc `Conv2D(64, 3)` + `MaxPooling2D(2)` combine ces motifs simples en motifs plus grands (un œil, une bouche).
4. `Flatten` puis `Dense(7, activation="softmax")` transforment ces motifs en **7 probabilités**, comme pour la baseline.

La compilation est identique à celle de la baseline (même perte, même optimiseur, même métrique) : les deux modèles seront donc comparables.

In [ ]:
# Partie 3a : définir et compiler le CNN de base
cnn = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (1,)),          # (48, 48, 1) : une image, 1 canal
        layers.Conv2D(32, 3, activation="relu"),       # 32 filtres 3x3 -> (46, 46, 32)
        layers.MaxPooling2D(2),                        # max sur chaque carré 2x2 -> (23, 23, 32)
        layers.Conv2D(64, 3, activation="relu"),       # 64 filtres 3x3 -> (21, 21, 64)
        layers.MaxPooling2D(2),                        # -> (10, 10, 64)
        layers.Flatten(),                              # 10 x 10 x 64 = 6400 valeurs à la suite
        layers.Dense(NUM_CLASSES, activation="softmax"),  # 7 probabilités qui somment à 1
    ],
    name="cnn_base",
)

cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Exemple : la 1re convolution a 3 x 3 x 1 x 32 poids + 32 biais = 320 paramètres,
# contre 295 040 pour la couche cachée de la baseline : les filtres sont partagés sur toute l'image.
cnn.summary()

### Comment un CNN lit une image : filtres, convolution, pooling

**1. Filtre et taille du noyau (*kernel size*).** Un filtre est une petite grille de poids, ici de 3 × 3 (`kernel_size=3`). C'est la plus petite taille qui a un centre et des voisins de tous les côtés ; elle suffit pour repérer un bord ou un coin.

**2. Convolution.** Le filtre glisse sur l'image. À chaque position, on multiplie ses 9 poids par les 9 pixels qu'il recouvre, on additionne le tout et on ajoute un biais. Les **mêmes poids** servent à toutes les positions : le motif est reconnu où qu'il soit, et il y a très peu de paramètres à apprendre.

**3. Carte de caractéristiques (*feature map*).** Le résultat d'un filtre est une nouvelle « image » qui indique où son motif a été trouvé. 32 filtres donnent 32 cartes, d'où la forme `(46, 46, 32)`. Dans la seconde convolution, chaque filtre regarde les 32 cartes à la fois : il a 3 × 3 × 32 poids.

**4. Pas (*stride*).** C'est le déplacement du filtre entre deux positions. Ici 1 pixel pour les convolutions (valeur par défaut) : on ne saute aucune position.

**5. *Padding*.** Avec `padding="valid"` (valeur par défaut), on n'ajoute rien autour de l'image : le filtre reste à l'intérieur, donc la sortie rétrécit de 2 pixels (48 − 3 + 1 = 46). Avec `padding="same"`, on entourerait l'image de zéros pour garder 48. Formule générale :

$$\text{taille de sortie} = \frac{\text{entrée} - \text{noyau} + 2 \times \text{padding}}{\text{pas}} + 1$$

**6. ReLU.** $\max(0, z)$ est appliqué à chaque valeur des cartes : on ne garde que les endroits où le motif est présent, et le réseau devient non linéaire (comme en Partie 2).

**7. *MaxPooling*.** `MaxPooling2D(2)` découpe chaque carte en carrés de 2 × 2 et ne garde que la plus grande valeur : 46 → 23, puis 21 → 10 (21 est impair, la dernière ligne et la dernière colonne sont ignorées). Il n'a aucun paramètre. Il réduit les calculs et rend la détection moins sensible à un décalage d'un pixel.

**8. `Flatten`, `Dense` et sortie.** Les 64 cartes de 10 × 10 sont mises bout à bout (6 400 valeurs), puis `Dense(7)` + softmax donne les 7 probabilités, comme pour la baseline.

**Tableau couche par couche** (à comparer avec `cnn.summary()` ci-dessus) :

| Couche | Forme de sortie | Paramètres | Calcul |
|---|---|---:|---|
| Entrée | (48, 48, 1) | 0 | |
| `Conv2D(32, 3)` + ReLU | (46, 46, 32) | 320 | (3 × 3 × 1 + 1) × 32 |
| `MaxPooling2D(2)` | (23, 23, 32) | 0 | |
| `Conv2D(64, 3)` + ReLU | (21, 21, 64) | 18 496 | (3 × 3 × 32 + 1) × 64 |
| `MaxPooling2D(2)` | (10, 10, 64) | 0 | |
| `Flatten` | (6400,) | 0 | 10 × 10 × 64 |
| `Dense(7)` + softmax | (7,) | 44 807 | 6 400 × 7 + 7 |
| **Total** | | **63 623** | |

**Pourquoi cette architecture ?**
- C'est la structure proposée par l'énoncé : un point de départ simple, que l'on peut expliquer couche par couche.
- Deux blocs suffisent pour des images de 48 × 48 : après deux *poolings*, les cartes ne font plus que 10 × 10.
- On double le nombre de filtres (32 → 64) quand la taille des cartes diminue : les motifs deviennent plus variés à mesure qu'ils sont plus grands.
- Le CNN a environ **4,6 fois moins de paramètres** que la baseline dense (63 623 contre 295 943), tout en tenant compte du voisinage des pixels.
- Les variantes (plus de filtres, couche `Dense` cachée, *dropout*…) seront testées une par une en Partie 6.

## Partie 4 — Entraînement du CNN

### 4a — Entraîner le CNN avec arrêt automatique

On entraîne le CNN comme la baseline, avec une différence : l'**arrêt automatique** (*early stopping*). À la fin de chaque époque, Keras regarde `val_loss`. Si elle ne s'est pas améliorée depuis 3 époques (`patience=3`), l'entraînement s'arrête et on reprend les poids de la **meilleure époque** (`restore_best_weights=True`).

**Pourquoi ?** Passé un certain point, le modèle apprend par cœur les images d'entraînement : `loss` continue de baisser mais `val_loss` remonte. Le modèle qu'on veut garder est celui d'avant ce point. `EPOCHS_CNN` n'est donc qu'un maximum.

In [ ]:
# Partie 4a : entraîner le CNN (le jeu de test n'est PAS utilisé ici)
EPOCHS_CNN = 30  # maximum : l'arrêt automatique stoppe en général bien avant

# Si val_loss ne s'améliore plus pendant 3 époques, on arrête
# et on reprend les poids de la meilleure époque (avant le surapprentissage).
arret_automatique = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

historique_cnn = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=[arret_automatique],
)

perte_val_cnn, accuracy_val_cnn = cnn.evaluate(val_ds, verbose=0)
print(f"CNN — accuracy sur la validation            : {accuracy_val_cnn:.1%}")
print(f"Baseline dense — accuracy sur la validation : {accuracy_val:.1%}")

## Partie 5 — Évaluation du CNN

Toute cette partie utilise la **validation**. Le jeu de test reste réservé à l'évaluation finale du modèle retenu, une seule fois.

### 5a — Matrice de confusion et rappel par classe

L'accuracy donne un seul chiffre. Comme les classes sont déséquilibrées, un modèle peut avoir une accuracy correcte en ratant presque toujours `disgust`. La **matrice de confusion** montre le détail :
- chaque **ligne** est une vraie classe, chaque **colonne** une classe prédite ;
- la **diagonale** compte les bonnes réponses ; toutes les autres cases sont des erreurs, et on voit **avec quelle classe** le modèle confond.

Le **rappel** (*recall*) d'une classe est la part de ses images que le modèle a retrouvées : case de la diagonale ÷ total de la ligne.

In [ ]:
# Partie 5a : matrice de confusion du CNN sur la validation
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# On parcourt la validation une seule fois : pour chaque lot, on garde les vraies
# étiquettes et les probabilités prédites. Les deux listes restent ainsi dans le même ordre.
labels_lots = []
probabilites_lots = []
for images, labels in val_ds:
    labels_lots.append(labels.numpy())
    probabilites_lots.append(cnn.predict_on_batch(images))
labels_onehot = np.concatenate(labels_lots)            # (5741, 7) : étiquettes one-hot
probabilites_cnn = np.concatenate(probabilites_lots)   # (5741, 7) : probabilités du CNN

vraies_classes = labels_onehot.argmax(axis=1)       # position du 1 = numéro de la vraie classe
classes_predites = probabilites_cnn.argmax(axis=1)  # classe la plus probable selon le CNN

matrice = confusion_matrix(vraies_classes, classes_predites)
ConfusionMatrixDisplay(matrice, display_labels=CLASS_NAMES).plot(cmap="Blues", xticks_rotation=45)
plt.title("CNN — matrice de confusion (validation)")
plt.show()

# Rappel par classe : bonnes réponses de la ligne / nombre d'images de cette classe
for numero, classe in enumerate(CLASS_NAMES):
    total_classe = matrice[numero].sum()
    print(f"{classe:10s} rappel = {matrice[numero, numero] / total_classe:.1%}  ({matrice[numero, numero]} / {total_classe})")

### 5b — Courbes ROC (une classe contre toutes les autres)

Une courbe ROC se trace pour une question à deux réponses. Avec 7 classes, on en trace donc 7 : pour chacune, « l'image est-elle de cette classe, **oui ou non** ? » (*one-vs-rest*).

Pour une classe, par exemple `happy`, on décide « oui » quand la probabilité prédite dépasse un **seuil**. En faisant varier ce seuil de 1 à 0, on obtient pour chaque valeur :
- le **taux de vrais positifs** (axe vertical) : part des vraies images `happy` reconnues — c'est le rappel ;
- le **taux de faux positifs** (axe horizontal) : part des autres images prises à tort pour `happy`.

Plus la courbe monte vite vers le coin en haut à gauche, mieux la classe est séparée des autres. L'**AUC** (aire sous la courbe) résume la courbe en un nombre : 1,0 = séparation parfaite, 0,5 = hasard (la diagonale en pointillés).

**Attention :** l'AUC mesure si le modèle donne des probabilités plus élevées aux bonnes images, pas s'il prédit finalement la bonne classe. Une classe rare comme `disgust` peut avoir une AUC élevée et un rappel faible : il faut lire les courbes ROC **avec** la matrice de confusion.

In [ ]:
# Partie 5b : courbes ROC du CNN sur la validation, une courbe par classe
from sklearn.metrics import auc, roc_curve

plt.figure(figsize=(7, 6))
for numero, classe in enumerate(CLASS_NAMES):
    # labels_onehot[:, numero] vaut 1 si l'image est de cette classe, 0 sinon ;
    # probabilites_cnn[:, numero] est la probabilité que le CNN donne à cette classe.
    taux_faux_positifs, taux_vrais_positifs, seuils = roc_curve(
        labels_onehot[:, numero], probabilites_cnn[:, numero]
    )
    aire = auc(taux_faux_positifs, taux_vrais_positifs)
    plt.plot(taux_faux_positifs, taux_vrais_positifs, label=f"{classe} (AUC = {aire:.2f})")

plt.plot([0, 1], [0, 1], "k--", label="hasard (AUC = 0.50)")
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs (rappel)")
plt.title("CNN — courbes ROC, une classe contre les autres (validation)")
plt.legend()
plt.show()